# Week 61, Day 1: with and why it exists

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w061d1.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
with open("notes.txt", "w") as f:
    f.write("first line\n")
print(f.closed)

def read_first(path):
    with open(path) as g:
        return g.readline().strip()        # returning from inside the block

print(read_first("notes.txt"))
try:
    with open("notes.txt") as h:
        raise ValueError("oops")
except ValueError:
    print("caught, and closed:", h.closed)

In [ ]:
log = []

def without_with(fail):
    log.append("open")
    try:
        if fail:
            raise RuntimeError("problem")
        return "done"
    finally:
        log.append("close")              # runs on the return and on the error

print(without_with(False))
try:
    without_with(True)
except RuntimeError:
    pass
print(log)

In [ ]:
class Announce:
    def __init__(self, name):
        self.name = name
    def __enter__(self):
        print("enter", self.name)
        return self
    def __exit__(self, exc_type, exc, tb):
        print("exit", self.name, exc_type.__name__ if exc_type else None)
        return False                      # don't suppress

with Announce("calm"):
    print("  body")

try:
    with Announce("stormy"):
        print("  body")
        1 / 0
        print("  never reached")
except ZeroDivisionError:
    print("the error carried on")

In [ ]:
class Step:
    def __init__(self, name):
        self.name = name
    def __enter__(self):
        print("enter", self.name)
        return self.name
    def __exit__(self, *exc):
        print("exit", self.name)

with Step("A") as a, Step("B") as b, Step("C") as c:
    print("  using", a, b, c)

In [ ]:
from decimal import Decimal, localcontext
import io

with localcontext() as ctx:
    ctx.prec = 3
    print(Decimal(1) / Decimal(7))
print(Decimal(1) / Decimal(7))                 # the default precision is back

with io.StringIO() as buffer:
    buffer.write("in memory")
    print(buffer.getvalue())
print(buffer.closed)

> **Key idea.** Resources must be released on every way out of a block: the end, return, break, and any exception. The with statement guarantees it: it evaluates the manager, calls __enter__ (binding its result to the as name), runs the body, then calls __exit__ with None three times on success or with the exception's type, value and traceback on an error. A true value from __exit__ suppresses the error; None lets it continue. If __enter__ fails, __exit__ isn't called. Several managers in one with are entered left to right and exited right to left. open, io.StringIO, decimal.localcontext, locks and database connections are all context managers.

### Exercise: Writing and reading safely

Write `save_lines(path, lines)`: write each line followed by `"\n"` to the file, with `with`, and return the number of lines; and `load_lines(path)`: the lines without their newlines, also with `with`. Then `append_line(path, line)`: add one line at the end (mode `"a"`). No explicit `close()` anywhere.

In [ ]:
def save_lines(path, lines):
    ___

def load_lines(path):
    ___

def append_line(path, line):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_lines():\n    \"with and files\"\n    assert save_lines(\"shop.txt\", [\"tea\", \"jam\"]) == 2 and load_lines(\"shop.txt\") == [\"tea\", \"jam\"]\n    append_line(\"shop.txt\", \"milk\")\n    assert load_lines(\"shop.txt\") == [\"tea\", \"jam\", \"milk\"]\n    assert save_lines(\"empty.txt\", []) == 0 and load_lines(\"empty.txt\") == []\n    assert \".close(\" not in source and source.count(\"with open\") >= 3")

### Exercise: with, by hand

Write `use_manager(manager, body)`: do exactly what `with manager as value: result = body(value)` does, **without** a `with` statement: call `__enter__`, pass its result to `body`, call `__exit__` with `(None, None, None)` on success or with the exception's type, value and traceback on an error, suppress the error if `__exit__` returns a true value (then return `None`), and otherwise re-raise it. On success return the body's result.

In [ ]:
def use_manager(manager, body):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("class Recorder:\n    def __init__(self, suppress=False):\n        self.log, self.suppress = [], suppress\n    def __enter__(self):\n        self.log.append(\"enter\")\n        return \"the value\"\n    def __exit__(self, t, e, tb):\n        self.log.append((\"exit\", t.__name__ if t else None, str(e) if e else None, tb is not None))\n        return self.suppress\ndef test_desugar():\n    \"the protocol\"\n    r = Recorder()\n    assert use_manager(r, lambda v: v.upper()) == \"THE VALUE\" and r.log == [\"enter\", (\"exit\", None, None, False)]\n    r = Recorder()\n    def bad(v):\n        raise KeyError(\"k\")\n    try:\n        use_manager(r, bad)\n        assert False\n    except KeyError:\n        pass\n    assert r.log == [\"enter\", (\"exit\", \"KeyError\", \"'k'\", True)]\n    r = Recorder(suppress=True)\n    assert use_manager(r, bad) is None and r.log[-1][1] == \"KeyError\"\n    class FailsToEnter:\n        def __enter__(self):\n            raise OSError(\"no\")\n        def __exit__(self, *a):\n            raise AssertionError(\"must not be called\")\n    try:\n        use_manager(FailsToEnter(), lambda v: v)\n        assert False\n    except OSError:\n        pass\n    assert \"with \" not in source")

### Exercise: Two files at once

Write `copy_upper(source, target)`: read the source file and write its contents in capitals to the target, using **one** `with` statement that opens both. Return the number of characters written. Then `merge_files(paths, target)`: write the lines of each file in turn to the target, opening each source file in its own `with` inside the target's.

In [ ]:
def copy_upper(source, target):
    ___

def merge_files(paths, target):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_copy():\n    \"several managers\"\n    with open(\"a.txt\", \"w\") as f:\n        f.write(\"tea\\njam\\n\")\n    with open(\"b.txt\", \"w\") as f:\n        f.write(\"milk\\n\")\n    assert copy_upper(\"a.txt\", \"A.txt\") == 8\n    with open(\"A.txt\") as f:\n        assert f.read() == \"TEA\\nJAM\\n\"\n    merge_files([\"a.txt\", \"b.txt\"], \"all.txt\")\n    with open(\"all.txt\") as f:\n        assert f.read() == \"tea\\njam\\nmilk\\n\"\n    assert \", open(\" in source.split(\"def copy_upper\")[1].split(\"def merge_files\")[0]")

### Exercise: Precision for a moment

Write `precise_divide(a, b, digits)`: `Decimal(a) / Decimal(b)` computed with `digits` significant digits, using `decimal.localcontext`, returned as a `Decimal`. The global precision must be unchanged afterwards. Then `banker_round(value, places)`: round a decimal string to `places` places with `ROUND_HALF_EVEN` inside a local context (use `Decimal.quantize`), returning a string.

In [ ]:
from decimal import Decimal, localcontext, ROUND_HALF_EVEN, getcontext

def precise_divide(a, b, digits):
    ___

def banker_round(value, places):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("from decimal import Decimal, getcontext\ndef test_decimal():\n    \"localcontext\"\n    before = getcontext().prec\n    assert precise_divide(1, 7, 3) == Decimal(\"0.143\") and precise_divide(2, 3, 10) == Decimal(\"0.6666666667\")\n    assert getcontext().prec == before\n    assert banker_round(\"2.5\", 0) == \"2\" and banker_round(\"3.5\", 0) == \"4\" and banker_round(\"1.005\", 2) == \"1.00\" and banker_round(\"1.015\", 2) == \"1.02\"\n    assert \"localcontext\" in source")

### Exercise: Entering and leaving

Write a context manager class `Tracker(name, log)` that appends `"enter NAME"` to the shared list `log` in `__enter__` (returning the name) and `"exit NAME"` in `__exit__`, adding `" after ERRORTYPE"` when there's an error (and not suppressing it). Then `nested_order()`: use `Tracker`s for the names `"a"`, `"b"` and `"c"` in **one** `with` statement, with the body appending `"body"` to the log, and return the log.

In [ ]:
class Tracker:
    def __init__(self, name, log):
        ___

def nested_order():
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_order():\n    \"left to right in, right to left out\"\n    assert nested_order() == [\"enter a\", \"enter b\", \"enter c\", \"body\", \"exit c\", \"exit b\", \"exit a\"]\n    log = []\n    try:\n        with Tracker(\"x\", log), Tracker(\"y\", log):\n            raise ValueError\n    except ValueError:\n        pass\n    assert log == [\"enter x\", \"enter y\", \"exit y after ValueError\", \"exit x after ValueError\"]\n    with Tracker(\"z\", log) as name:\n        assert name == \"z\"")

### Bug hunt: Reading after the block

The file is closed when the `with` block ends, so reading it afterwards raises `ValueError: I/O operation on closed file`. Read inside the block so this prints `hello`.

In [ ]:
with open("greeting.txt", "w") as f:
    f.write("hello")

with open("greeting.txt") as f:
    pass
try:
    print(f.read())
except ValueError:
    print("closed")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"inside the block\"\n    expect_output(\"hello\")")

### Bug hunt: A manager that swallows errors

`__exit__` returns `True`, so **every** error in the block is silently suppressed, including the real bug, and the program prints `finished` as if all were well. Return `False` (or nothing) so the error reaches the `except` and this prints `ZeroDivisionError`.

In [ ]:
class Quiet:
    def __enter__(self):
        return self
    def __exit__(self, exc_type, exc, tb):
        return True

try:
    with Quiet():
        total = 10 / 0
    print("finished")
except ZeroDivisionError:
    print("ZeroDivisionError")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"don't suppress\"\n    expect_output(\"ZeroDivisionError\")")

### Bug hunt: The as value is __enter__'s

`as` receives what `__enter__` **returns**. This `__enter__` sets up a connection but forgets to return it, so `conn` is `None` and calling `conn.send` raises `AttributeError`. Return the connection so it prints `sent: hi`.

In [ ]:
class Connection:
    def send(self, text):
        return f"sent: {text}"

class Connect:
    def __enter__(self):
        self.conn = Connection()
    def __exit__(self, *exc):
        self.conn = None

try:
    with Connect() as conn:
        print(conn.send("hi"))
except AttributeError:
    print("AttributeError")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"return from __enter__\"\n    expect_output(\"sent: hi\")")

### Puzzle: What with does

Arrange the lines into a hand-written version of `with manager as value: body(value)`. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # True or False: __exit__ is called if __enter__ raises
q = ___   # the three arguments __exit__ receives when the body finishes normally, as a tuple
r = ___   # with A(), B(): ... which manager is exited first: "A" or "B"
s = ___   # True or False: returning True from __exit__ suppresses an exception
t = ___   # the exception from reading a file after its with block, as a string

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p is False and q == (None, None, None) and r == \"B\", \"p, q and r\"\n    assert s is True and t == \"ValueError\", \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: with A, B, C, by hand

Write `run_all(managers, body)`: do exactly what `with m1 as v1, m2 as v2, ...: result = body([v1, v2, ...])` does, without a `with` statement, for any number of managers (including none):

- Enter the managers left to right, collecting their values. If entering one raises, exit the ones already entered (right to left) with that exception, then let it propagate (unless one of them suppresses it).
- Run the body with the list of values.
- Exit the managers right to left. Each `__exit__` receives the exception that's **current** at that point: the body's (or a later manager's) exception, or `(None, None, None)` if there is none. If an `__exit__` returns true, the exception is cleared, so the managers exited after it see `None`s. If an `__exit__` raises, that new exception replaces the current one.
- At the end, re-raise the current exception if there is one; otherwise return the body's result (or `None` if the body's error was suppressed).

(Hint: a recursive helper that enters one manager and calls itself for the rest makes this almost a direct translation of nested `with` statements, which is exactly what `with a, b:` means.)

In [ ]:
def run_all(managers, body):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("class M:\n    def __init__(self, name, log, suppress=False, fail_enter=False, fail_exit=False):\n        self.name, self.log, self.suppress, self.fail_enter, self.fail_exit = name, log, suppress, fail_enter, fail_exit\n    def __enter__(self):\n        if self.fail_enter:\n            self.log.append(f\"enter {self.name} failed\")\n            raise OSError(self.name)\n        self.log.append(f\"enter {self.name}\")\n        return self.name\n    def __exit__(self, t, e, tb):\n        self.log.append(f\"exit {self.name} {t.__name__ if t else None}\")\n        if self.fail_exit:\n            raise RuntimeError(f\"exit {self.name}\")\n        return self.suppress\ndef compare(build, body):\n    log1, log2 = [], []\n    def run_real(log):\n        ms = build(log)\n        try:\n            result = None\n            if len(ms) == 3:\n                with ms[0] as a, ms[1] as b, ms[2] as c:\n                    result = body([a, b, c])\n            else:\n                with ms[0] as a, ms[1] as b:\n                    result = body([a, b])\n            return (\"ok\", result)\n        except BaseException as e:\n            return (\"err\", type(e).__name__, str(e))\n    def run_mine(log):\n        try:\n            return (\"ok\", run_all(build(log), body))\n        except BaseException as e:\n            return (\"err\", type(e).__name__, str(e))\n    r1, r2 = run_real(log1), run_mine(log2)\n    assert (r1, log1) == (r2, log2), (r1, log1, r2, log2)\ndef test_run_all():\n    \"the full semantics\"\n    ok = lambda vals: \"+\".join(vals)\n    def boom(vals):\n        raise ValueError(\"body\")\n    compare(lambda log: [M(\"a\", log), M(\"b\", log), M(\"c\", log)], ok)\n    compare(lambda log: [M(\"a\", log), M(\"b\", log), M(\"c\", log)], boom)\n    compare(lambda log: [M(\"a\", log), M(\"b\", log, suppress=True), M(\"c\", log)], boom)\n    compare(lambda log: [M(\"a\", log), M(\"b\", log, fail_enter=True), M(\"c\", log)], ok)\n    compare(lambda log: [M(\"a\", log, suppress=True), M(\"b\", log, fail_enter=True), M(\"c\", log)], ok)\n    compare(lambda log: [M(\"a\", log), M(\"b\", log, fail_exit=True), M(\"c\", log)], ok)\n    compare(lambda log: [M(\"a\", log, suppress=True), M(\"b\", log, fail_exit=True)], boom)\n    assert run_all([], lambda vals: len(vals)) == 0\n    assert \"with \" not in source")

## Warm-up

Whichever way the code leaves the `with` block (the end, a `return`, or an exception), the file is closed. That guarantee is the whole point.

## Every way out

Code that uses a resource has to give it back, and there are many ways out of a block: falling off the end, `return`, `break` and `continue` in a loop, and any exception raised anywhere inside, including by functions it calls. Missing one leaks the resource: files stay open (and their data may not be written), locks stay held (and the program freezes), temporary settings stay changed.

Week 11's tool for this was `try`/`finally`: the `finally` block runs on every path. It works, but every use repeats the same three parts (acquire, `try`, release in `finally`), and it's easy to acquire inside the `try` by mistake, or forget it entirely. `with` turns the pattern into a reusable object: the object knows how to set up and how to clean up, and the statement guarantees both happen.

## What with really does

A **context manager** is any object with two methods: `__enter__(self)` and `__exit__(self, exc_type, exc_value, traceback)`. The statement

```python
with EXPRESSION as name:
    BODY
```

does, in effect (from PEP 343, by Guido van Rossum and Nick Coghlan, 2005):

```python
manager = EXPRESSION
name = manager.__enter__()
try:
    BODY
except BaseException as error:
    if not manager.__exit__(type(error), error, error.__traceback__):
        raise                      # __exit__ returned a false value: let the error carry on
else:
    manager.__exit__(None, None, None)
```

So:

- `__enter__` runs **before** the body. If it raises, the body never runs and `__exit__` isn't called (nothing was entered).
- `__exit__` runs **after** the body, however it ends. With no error, its three arguments are `None`. With an error, they describe it.
- If `__exit__` returns a **true** value, the error is **suppressed**: execution continues after the `with` as if nothing happened. Returning `None` (the usual case) lets the error carry on. Suppressing errors should be rare and deliberate.

## The as value

`as name` receives whatever `__enter__` **returns**, which isn't always the manager itself. `open(...)` returns a file whose `__enter__` returns the same file, so `f` is the file. But a manager can hand out something else: a lock's `__enter__` returns `True`, `decimal.localcontext()`'s returns the new context, and tomorrow you'll write managers that return a helper object. `as` is optional: `with lock:` is common when the value isn't needed.

The name stays bound **after** the block (like a `for` loop variable), so `f` still refers to the (now closed) file. Using it for reading or writing then raises `ValueError: I/O operation on closed file`.

## Several at once

`with A() as a, B() as b:` is the same as nesting `with B()` inside `with A()`: managers are entered **left to right** and exited **right to left**, so whatever was set up last is cleaned up first. If entering B fails, A is still exited properly. Since Python 3.10 you can put them in brackets over several lines:

```python
with (
    open("in.txt") as source,
    open("out.txt", "w") as target,
):
    ...
```

## Ready-made managers

Context managers you can use already:

- **Files** from `open`, and many file-like objects: `io.StringIO`, `zipfile.ZipFile`, `tempfile.TemporaryDirectory()`.
- **Temporary settings**: `decimal.localcontext()` changes decimal precision or rounding just for the block.
- **Locks** from `threading` (Week 66): `with lock:` holds the lock for exactly the block.
- **Database connections and transactions**, such as `sqlite3` (Phase 6): commit on success, roll back on error.
- **`contextlib`'s helpers** (Wednesday): `suppress`, `redirect_stdout`, `chdir` and the `contextmanager` decorator.

## Practice

## Quiz

## Daily challenge

## Before you go

List three things in programs you've written that had to be "put back" afterwards (a file, a setting, a temporary change), and say how each was released. Tomorrow: writing your own context manager classes.